[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/bloc3_ml/corrections/seance1_correction.ipynb)

# Séance 3.1 — Prédire un nombre, prédire une décision

**Correction** · durée : 2h (≈55 min de cours, ≈45 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/Intelligence-Artificielle-et-Data-Science/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- dire ce qui sépare un modèle qui explique d'un modèle qui prédit
- découper en apprentissage et test, et dire pourquoi c'est la seule note qui compte
- mesurer une erreur en euros (MAE, RMSE, R²) et repérer une fuite de données
- ajuster une régression logistique et lire une matrice de confusion
- choisir un seuil de décision à partir d'un coût, pas d'une habitude

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/Intelligence-Artificielle-et-Data-Science/main/bloc3_ml/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
# --- Moitie 1 : predire un NOMBRE, chez le detaillant --------------------
cmd = pd.read_csv(BASE + "commandes.csv")

Xr = cmd[["qte", "nart"]]   ## ce qu'on connait avant de facturer
yr = cmd["ca"]              ## ce qu'on veut prevoir

Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(Xr, yr, test_size=0.25, random_state=67)

# --- Moitie 2 : predire une DECISION, chez l'operateur telecom -----------
tel = pd.read_csv(BASE + "churn.csv")
tel["total"] = pd.to_numeric(tel["total"], errors="coerce")
tel = tel.dropna(subset=["total"])

yc = tel["churn"]
Xc = pd.get_dummies(tel.drop(columns=["churn"]), drop_first=True)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    Xc, yc, test_size=0.3, random_state=42, stratify=yc)

print(len(Xr_tr), "commandes d'apprentissage |", len(Xc_tr), "abonnes d'apprentissage")

---

# Partie 1 — L'échauffement

Le code est déjà écrit : il ne reste que les `____` à remplir. Allez vite, l'essentiel
de la séance est dans la partie 2.

### Exercice 1 — Ajuster et prédire

> **Votre mission :**
> - Ajuster une régression linéaire sur le jeu d'**apprentissage** → `reg`.
> - Prédire sur le jeu de **test** → `pr`.
> - Mettre la première prédiction, arrondie à 2 décimales, dans `p1`.

In [ ]:
# On apprend sur Xr_tr / yr_tr, on predit sur Xr_te : jamais l'inverse
reg = LinearRegression().fit(Xr_tr, yr_tr)   ## apprendre
pr = reg.predict(Xr_te)                      ## noter, sur des lignes jamais vues

p1 = round(pr[0], 2)   ## une prediction par ligne de test
print(len(pr), "predictions | la premiere :", p1)

In [ ]:
verifier("1a - nombre de predictions", len(pr) == 489, "on predit sur le jeu de test")
verifier("1b - premiere prediction", abs(p1 - 370.99) < 1,
         "ajustez sur Xr_tr et yr_tr, puis predisez sur Xr_te")

### Exercice 2 — L'erreur en euros, et ce que la RMSE voit de plus

> **Votre mission :**
> - Calculer la MAE et la RMSE **sur le jeu de test** → `mae` et `rmse`, arrondies à 1 décimale. La RMSE est la racine de `mean_squared_error`.
> - Elles ne sont pas égales : calculer leur rapport → `rapport` (2 décimales).
> - Puis mesurer ce que pèsent les **5 % de commandes les plus mal prédites** dans le total des erreurs → `part_pires` (1 décimale, en %).
> - *Nouveau :* `serie.nlargest(k)` garde les `k` plus grandes valeurs.

In [ ]:
mae = round(mean_absolute_error(yr_te, pr), 1)   ## en euros

# ** 0.5 : la racine carree. RMSE = Root Mean Squared Error
rmse = round(mean_squared_error(yr_te, pr) ** 0.5, 1)
rapport = round(rmse / mae, 2)

ecarts = (yr_te - pr).abs()
k = int(0.05 * len(ecarts))
part_pires = round(100 * ecarts.nlargest(k).sum() / ecarts.sum(), 1)

print("MAE", mae, "| RMSE", rmse, "| rapport", rapport, "| 5 % pires :", part_pires, "%")

# La RMSE vaut 2,68 fois la MAE : elle eleve les erreurs au CARRE avant de
# les moyenner, donc une faute de 2 000 EUR y pese cent fois une faute de
# 200 EUR. Et ca se verifie : 5 % des commandes portent 44,3 % de l'erreur
# totale. La MAE seule cachait une poignee de devis tres faux.

In [ ]:
verifier("2a - MAE", abs(mae - 217.5) < 2, "mean_absolute_error(yr_te, pr)")
verifier("2b - RMSE", abs(rmse - 583.3) < 2, "la racine carree s'ecrit ** 0.5")
verifier("2c - le rapport RMSE/MAE", abs(rapport - 2.68) < 0.05, "rmse divise par mae")
verifier("2d - le poids des 5 % pires", abs(part_pires - 44.3) < 1,
         "nlargest(k) garde les k plus grandes erreurs")

### Exercice 3 — Les deux notes du même modèle

> **Votre mission :**
> - Calculer le R² **en apprentissage** → `r2_tr`, et **en test** → `r2_te`, arrondis à 3 décimales.
> - Lequel des deux est le meilleur ? De combien ? Cet écart est-il inquiétant ?

In [ ]:
r2_tr = round(r2_score(yr_tr, reg.predict(Xr_tr)), 3)   ## sur le vu
r2_te = round(r2_score(yr_te, reg.predict(Xr_te)), 3)   ## sur le jamais vu

print("apprentissage", r2_tr, "| test", r2_te, "| ecart", round(r2_tr - r2_te, 3))

# L'apprentissage est meilleur, comme toujours : les coefficients ont ete
# calcules pour coller a CES lignes-la. Quatre points d'ecart, c'est le
# prix normal a payer. Un ecart de quarante points, lui, s'appellerait du
# surapprentissage : le modele aurait retenu au lieu d'apprendre.

In [ ]:
verifier("3a - R2 en apprentissage", abs(r2_tr - 0.732) < 0.02, "predisez sur Xr_tr")
verifier("3b - R2 en test", abs(r2_te - 0.691) < 0.02, "predisez sur Xr_te")

### Exercice 4 — La logistique, et le piège de la justesse

> **Votre mission :**
> - On passe à la seconde moitié. Construire un pipeline `StandardScaler` puis `LogisticRegression(max_iter=1000)` → `clf`, l'ajuster sur l'apprentissage, et récupérer les probabilités de départ du test → `proba`.
> - Calculer la justesse du modèle → `just_modele` (en %, 1 décimale).
> - Puis celle d'un modèle qui prédit que **personne** ne part → `just_nul`. Combien de points le modèle gagne-t-il réellement ?

In [ ]:
# make_pipeline enchaine les etapes : mise a l'echelle, puis modele
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
clf.fit(Xc_tr, yc_tr)   ## sur l'apprentissage uniquement

# predict_proba renvoie [proba de rester, proba de partir] : l'indice 1
proba = clf.predict_proba(Xc_te)[:, 1]

pred = clf.predict(Xc_te)   ## des 0 et des 1, tranches a 0,50
just_modele = round(100 * accuracy_score(yc_te, pred), 1)   ## 79,8 %

# La part de ceux qui restent : c'est la justesse du modele qui dit
# toujours "reste"
just_nul = round(100 * (1 - yc_te.mean()), 1)
print(just_modele, "% contre", just_nul, "% ->", round(just_modele - just_nul, 1), "points")

In [ ]:
verifier("4a - probabilites de depart", len(proba) == 2110 and proba.max() <= 1,
         "la colonne d'indice 1 est la probabilite de partir")
verifier("4b - justesse du modele", abs(just_modele - 79.8) < 1, "accuracy_score(yc_te, pred)")
verifier("4c - justesse du modele nul", abs(just_nul - 73.4) < 0.5,
         "c'est la proportion de clients qui restent")

### Exercice 5 — La matrice de confusion, et les trois taux qui en sortent

> **Votre mission :**
> - Afficher la matrice de confusion, et mettre dans `perdus` le nombre de clients **partis sans qu'on les ait détectés**. C'est la case qui coûte de l'argent.
> - Puis calculer la précision → `prec`, le rappel → `rapp` et le F1 → `f1`, arrondis à 3 décimales.
> - Vérifier que le F1 vaut bien `2 × prec × rapp / (prec + rapp)` → `f1_main`.
> - Traduire précision et rappel en une phrase de gestion, en commentaire.

In [ ]:
mat = confusion_matrix(yc_te, pred)
print(mat)   ## lignes : la verite | colonnes : la prediction

# Ligne 1 = ceux qui partent vraiment, colonne 0 = ceux qu'on predit
# comme restant. L'intersection : les partants qu'on n'a pas vus venir.
perdus = mat[1][0]

prec = round(precision_score(yc_te, pred), 3)   ## parmi ceux qu'on appelle
rapp = round(recall_score(yc_te, pred), 3)      ## parmi ceux qui partent
f1 = round(f1_score(yc_te, pred), 3)

# La formule du F1, refaite a la main : elle n'a rien de mysterieux
f1_main = round(2 * prec * rapp / (prec + rapp), 3)

print(perdus, "perdus | precision", prec, "| rappel", rapp, "| F1", f1, "| a la main", f1_main)

# Precision 0,64 : sur 10 abonnes contactes, 6 allaient vraiment partir.
# Rappel 0,545 : nous retrouvons un partant sur deux, l'autre s'en va.
# Les trois taux se lisent DANS la matrice : ce ne sont pas trois mesures
# de plus, ce sont trois facons de resumer les memes quatre cases.

In [ ]:
verifier("5a - clients perdus non detectes", abs(perdus - 255) <= 12,
         "ligne des vrais partants, colonne des predits restants")
verifier("5b - precision", abs(prec - 0.640) < 0.03, "precision_score")
verifier("5c - rappel", abs(rapp - 0.545) < 0.03, "la fonction s'appelle recall_score")
verifier("5d - F1 refait a la main", abs(f1 - f1_main) < 0.01,
         "2 fois le produit, divise par la somme")

### Exercice 6 — Descendre le seuil, et compter ce que ça rapporte

> **Votre mission :**
> - Décider à **0,30** au lieu de 0,50 → `pred30`, puis recalculer précision et rappel → `prec30`, `rapp30`. Lequel monte, lequel descend ?
> - Un seuil ne se choisit pas au jugé : il se chiffre. Un appel coûte **15 €**, un client retenu rapporte **300 €**, une relance en convainc **30 %**.
> - Écrire la fonction `gain(seuil)`, puis calculer le gain au seuil 0,50 → `gain50` et au seuil 0,20 → `gain20`.

In [ ]:
pred30 = (proba > 0.30).astype(int)   ## on appelle des 30 % de risque
prec30 = round(precision_score(yc_te, pred30), 3)   ## descend
rapp30 = round(recall_score(yc_te, pred30), 3)      ## monte
print("seuil 0.30 -> precision", prec30, "| rappel", rapp30)

def gain(seuil):
    p = (proba > seuil).astype(int)
    vrais = ((p == 1) & (yc_te == 1)).sum()     ## partants rattrapes
    return vrais * 0.30 * 300 - p.sum() * 15    ## 15 euros par appel passe

gain50, gain20 = gain(0.50), gain(0.20)
print(round(gain50), "euros contre", round(gain20), "euros")

# Le rappel monte, la precision descend : c'est toujours ce compromis, et
# aucune mesure statistique ne dit ou s'arreter. L'argent, si : 8 000 EUR
# d'ecart pour un seul nombre change.

In [ ]:
verifier("6a - rappel a 0,30", rapp30 > rapp, "un seuil plus bas retrouve plus de partants")
verifier("6b - precision a 0,30", prec30 < prec,
         "un seuil plus bas contacte plus de gens pour rien")
verifier("6c - gain au seuil 0,50", abs(gain50 - 20370) < 1200, "le cout d'un appel est 15")
verifier("6d - gain au seuil 0,20", abs(gain20 - 28365) < 1200, "meme calcul, seuil 0.20")

---

# Partie 2 — Les questions

Ici, plus de trous : **la cellule sous chaque question est vide**, et c'est à vous
d'écrire le code en entier. C'est exactement ce qu'on vous demandera pour le projet
final, et ce que fait un analyste devant un fichier qu'il découvre.

Certaines questions utilisent une commande que le cours n'a pas montrée. Quand c'est le
cas, l'énoncé vous la donne — savoir se servir d'une commande qu'on vient de lire fait
partie du métier.

> 💡 Pas de vérification automatique dans cette partie. Affichez systématiquement votre
> résultat, et demandez-vous s'il est **plausible** avant de passer à la suite : c'est
> le seul contrôle dont vous disposerez en entreprise.

### Question 7 — Lire ce que la régression a appris

> **Votre mission :**
> - Un modèle ajusté, ce sont des **nombres** qu'on peut lire. Relever la constante de `reg` → `constante`, puis ses deux coefficients → `coef_qte` et `coef_nart` (2 décimales).
> - *Indice :* `reg.intercept_` donne la constante, `reg.coef_` les coefficients dans l'ordre des colonnes de `Xr`.
> - Puis : que prédit ce modèle pour une commande de **0 unité et 0 produit** ? Cette valeur a-t-elle un sens commercial ?

In [ ]:
constante = round(reg.intercept_, 2)
coef_qte = round(reg.coef_[0], 2)     ## premiere colonne de Xr : qte
coef_nart = round(reg.coef_[1], 2)    ## deuxieme colonne : nart

print(f"ca = {constante} + {coef_qte} x qte + {coef_nart} x nart")

# 1,32 EUR par unite supplementaire, 3,64 EUR par produit distinct de plus.
# La constante vaut 62,86 : c'est ce que le modele predit pour une commande
# de zero unite et zero produit. Une telle commande n'existe pas — la
# constante POSITIONNE la droite, elle ne se lit pas comme un montant.
# Notez qu'elle a change depuis le cours (114,59 avec la seule qte) : un
# coefficient depend TOUJOURS des autres variables du modele.

### Question 8 — La fuite discrète

> **Votre mission :**
> - La fuite du cours était grossière : on donnait `ca` au modèle. En voici une réaliste — ajouter une colonne `remise`, qui vaut 5 % du montant facturé.
> - Rien dans son nom ne dit qu'elle contient la réponse — et pourtant.
> - Mesurer le R² de test et expliquer ce qui s'est passé.

In [ ]:
discret = cmd[["qte", "nart"]].copy()
discret["remise"] = (cmd["ca"] * 0.05).round(2)   ## 5 % du montant facture

a, b, c, d = train_test_split(discret, yr, test_size=0.25, random_state=67)
print("R2 :", round(r2_score(d, LinearRegression().fit(a, c).predict(b)), 3))

# R2 = 1,0. La remise vaut 5 % du montant : c'est la reponse, ecrite
# autrement. Aucune colonne ne s'appelait "ca", et pourtant la fuite est
# totale. Le test a appliquer : cette colonne serait-elle disponible AVANT
# que la commande soit facturee ? Non. Elle n'a rien a faire la.

### Question 9 — Le modèle le plus bête

> **Votre mission :**
> - Construire un modèle de référence qui prédit **toujours la même valeur** : la moyenne des montants d'apprentissage.
> - Calculer sa RMSE de test, et son R². Comparer au vrai modèle.
> - Vous venez de faire côté régression ce que l'exercice 4 faisait côté décision. Pourquoi est-ce le premier geste, toujours ?

In [ ]:
bete = np.full(len(yr_te), yr_tr.mean())   ## toujours la meme reponse

print("RMSE du modele bete :", round(mean_squared_error(yr_te, bete) ** 0.5, 1))
print("RMSE du vrai modele :", round(mean_squared_error(yr_te, pr) ** 0.5, 1))
print("R2 du modele bete   :", round(r2_score(yr_te, bete), 3))

# Le R2 du modele constant vaut zero a un millieme pres (RMSE 1 051 EUR,
# contre 583 pour le vrai modele) : c'est exactement ce que le R2 mesure —
# le gain par rapport a "toujours la moyenne".
#
# C'est le premier geste parce que c'est le seul qui donne une ECHELLE.
# "R2 = 0,69" ou "justesse = 79,8 %" ne veulent rien dire tant qu'on ne
# sait pas ce que score le modele qui ne fait rien.

### Question 10 — Les coefficients de la logistique

> **Votre mission :**
> - Extraire les coefficients de `clf` et les trier par valeur absolue décroissante.
> - Les variables ayant été mises à l'échelle, ils sont comparables entre eux.
> - Quelles sont les trois qui pèsent le plus, et dans quel sens ?
> - *Nouveau :* dans un pipeline, on accède à la dernière étape par `clf[-1]`.

In [ ]:
# clf[-1] : la derniere etape du pipeline, la logistique elle-meme
coefs = pd.Series(clf[-1].coef_[0], index=Xc.columns)

coefs.reindex(coefs.abs().sort_values(ascending=False).index).round(3).head(6)

# Un coefficient positif pousse au depart, negatif retient. Le contrat
# arrive en tete, et c'est coherent avec l'exercice 1 : 42,7 % de departs
# au contrat mensuel contre 2,8 % a deux ans.

### Question 11 — Le meilleur F1 est-il le meilleur seuil ?

> **Votre mission :**
> - Chercher le seuil qui maximise le **F1**, puis celui qui maximise le **gain** de la campagne.
> - Sont-ils au même endroit ? Combien coûte le fait de suivre le F1 plutôt que l'euro ?
> - *Nouveau :* `serie.idxmax()` donne l'indice de la plus grande valeur.

In [ ]:
seuils = np.arange(0.05, 1.0, 0.05).round(2)
f1s = pd.Series([f1_score(yc_te, (proba > s).astype(int)) for s in seuils], index=seuils)
gains = pd.Series([gain(s) for s in seuils], index=seuils)

print("meilleur F1   : seuil", f1s.idxmax(), "->", round(f1s.max(), 3))
print("meilleur gain : seuil", gains.idxmax(), "->", round(gains.max()), "euros")
print("gain au seuil choisi par le F1 :", round(gain(f1s.idxmax())), "euros")

# Le F1 place l'optimum a 0,35, l'argent a 0,20. Suivre le F1 couterait
# 2 500 EUR. C'est normal et il faut le comprendre : le F1 traite les deux
# erreurs comme si elles avaient le meme poids, alors qu'ici un appel
# inutile coute 15 EUR et un client perdu en coute 90. Une mesure
# statistique cadre la decision, elle ne la prend pas.

### Question 12 — Question de synthèse

> **Votre mission :**
> - Vous avez construit deux modèles, sur deux entreprises différentes. On vous demande **une note de cinq lignes** pour un comité de direction.
> - Pour le détaillant : l'erreur médiane en euros et sa part du panier médian. Peut-on automatiser le devis ?
> - Pour l'opérateur : le nombre d'appels au seuil retenu et le gain net. Faut-il lancer la campagne ?
> - Chaque réponse doit tenir en une phrase et porter un chiffre.

In [ ]:
err = (yr_te - pr).abs()
print("detaillant : erreur mediane", round(err.median(), 2), "euros sur un panier median de",
      round(yr_te.median(), 2))
print("           soit", round(100 * err.median() / yr_te.median(), 1), "%")

p20 = (proba > 0.20).astype(int)
print("operateur  :", int(p20.sum()), "appels ->", round(gain(0.20)), "euros de gain net")
print("           soit", round(gain(0.20) / p20.sum(), 2), "euros par appel")

# Note possible :
#
# "Devis automatique : NON en l'etat. Nous nous trompons de 91 EUR une fois
#  sur deux sur un panier median de 343 EUR, soit 27 %. C'est utilisable
#  pour dimensionner un stock, pas pour engager un client.
#
#  Campagne de retention : OUI. En appelant les 1 073 abonnes dont le
#  risque depasse 20 %, la campagne rapporte environ 28 000 EUR nets, soit
#  26 EUR par appel. Le seuil de 0,20 est volontairement bas parce qu'un
#  appel coute 15 EUR et un client perdu en coute 90 ; il est a revoir si
#  le cout d'un contact augmente."
#
# Remarquez ce que les deux reponses ont en commun : aucune ne cite un R2
# ni une justesse. Elles citent des euros et un volume d'action.